# Trees & Traversals: Zero to Hero

**NB-06 in the [DSA: Zero to Hero](README.md) series.**

The first structure in this series that is defined recursively — which makes recursion the natural
tool, and makes **recursion depth** the thing that breaks. All four traversals, each converted to
an explicit stack, plus Morris traversal in $O(1)$ space with its restoration property verified.

***

## Why this notebook is different

- **The recursion limit is measured, not warned about.** §3 builds a degenerate tree and finds the
  exact size where recursive traversal dies: **1,000 nodes** in CPython. Then it traverses a
  **1,000,000-node balanced tree recursively without trouble** — because the limit is on *depth*,
  not size, and $\log_2 10^6 \approx 20$. Those two measurements together are the whole lesson.
- **Java's limit is measured on the same axis.** A default JVM thread reaches **~23,000 frames**;
  give the thread 64 MB of stack and it reaches **millions**. Java's advantage over Python here is
  real — roughly 23× by default — and it is a difference of degree rather than kind.
- **Morris traversal is verified where it actually matters.** It walks a tree in $O(1)$ space by
  *temporarily rewiring it*, which is only safe if it puts everything back. §1.4 checks the tree is
  **structurally identical afterwards** across 3,000 randomised trees — the property that makes the
  algorithm trustworthy, and the one most treatments simply assert.

The through-line from NB-05: "stack for depth, queue for breadth" stops being a slogan here.
Swapping one container for the other in the same six-line function turns depth-first into
breadth-first, and §1.3 shows the recursive traversals *already contain* a stack — the call stack —
which is exactly what the explicit-stack conversion makes visible.

***

## Contents

**Part 1 — Theory from zero**
1. The recursive definition, and why recursion fits
2. The four traversals
3. **Converting recursion to an explicit stack**
4. **Morris traversal** — $O(1)$ space, and the property that makes it safe
5. The same traversals in Java, cross-checked

**Part 2 — Worked problems** — height and depth, diameter, BFS variants, serialise/deserialise
**Part 3 — The signature difficulty: recursion depth is a real constraint**
**Part 4 — Tough questions** · **Part 5 — Practice** · **Part 6 — Reading**

***

## In one paragraph

A **binary tree** is either empty or a node with a value and two subtrees — a definition that is
*recursive*, which is why almost every tree algorithm is naturally written as recursion and why the
base case is always "the empty tree". Visiting every node admits exactly four orders that matter:
**preorder** (node, left, right), **inorder** (left, node, right), **postorder** (left, right,
node) and **level order** (breadth-first). The first three differ only in *where you put one line*,
and each is the right answer to a different question — preorder for copying and serialising,
inorder for sorted output from a BST (NB-07), postorder for anything where children must be
finished before their parent, such as freeing memory or computing sizes. Level order is the odd one
out: it is not a recursion at all, it is a **queue** (NB-05), and swapping that queue for a stack
converts it back into depth-first, which is the cleanest demonstration in the series that these
structures are *policies*. Every recursive traversal can be rewritten with an **explicit stack** —
the conversion is mechanical for preorder, fiddly for postorder, and necessary whenever depth might
exceed the runtime's limit. **Morris traversal** goes further and uses $O(1)$ space by threading
temporary pointers through the tree and removing them again. The costs are all $\Theta(n)$ time;
what varies is space, which ranges from $\Theta(n)$ for the call stack on a degenerate tree down to
$\Theta(1)$ for Morris — and §3 measures why that range matters.

**Prerequisites:** [NB-04 Linked Lists](linked_lists_zero_to_hero.ipynb) for nodes and pointers and
§2.1's measured recursion limit, and [NB-05 Stacks, Queues & Deques](stacks_queues_zero_to_hero.ipynb)
for the stack and queue this notebook uses throughout — §1.1 there ("stack for depth, queue for
breadth") is the sentence this notebook makes concrete.

***
# Part 0 - Setup

Standard library only, plus `dsa_toolkit` from this folder. §1.5 and §3 need the JDK.

In [1]:
# ---------------------------------------------------------------------------
# Everything this notebook uses. Standard library only.
# ---------------------------------------------------------------------------
import math
import random
import sys
import time
from collections import deque

from dsa_toolkit import (InvariantError, JavaError, StressFailure, check_invariant,
                         cross_check, growth_table, java_available, measure_growth,
                         run_java, stress)

RANDOM_SEED = 12345

ok, detail = java_available()
JAVA = ok
print("python", sys.version.split()[0])
print("JDK available:", ok, "|", detail)
print("recursion limit:", sys.getrecursionlimit(), " <- Part 3 is about this number")

python 3.14.7
JDK available: True | javac 25.0.4.1
recursion limit: 1000  <- Part 3 is about this number


***
# Part 1 - Theory from zero

1. The recursive definition, and why recursion fits
2. The four traversals
3. **Converting recursion to an explicit stack**
4. **Morris traversal** — $O(1)$ space, and the property that makes it safe
5. The same traversals in Java, cross-checked

## 1.1 The recursive definition, and why recursion fits

Every structure so far has been defined by *layout* — an array is contiguous storage, a linked list
is nodes joined by pointers. A tree is defined by **recursion**:

> A binary tree is either **empty**, or a **node** holding a value and two binary trees.

That is the whole definition, and the shape of every algorithm in this notebook follows from it.
When your data is defined by cases, your functions are written by cases — one branch for empty, one
for a node — and the node branch calls itself on the two subtrees. The base case is never an
afterthought you remember to add; it is the first half of the definition.

A little vocabulary, because these are used inconsistently everywhere and the inconsistency causes
real bugs (§2.1):

| Term | Meaning here |
|---|---|
| **depth** of a node | edges from the **root** down to it; the root has depth 0 |
| **height** of a node | edges from it down to its deepest **leaf**; a leaf has height 0 |
| **height** of a tree | the height of its root |
| **height of the empty tree** | **−1**, by the convention this notebook uses |
| **level** | all nodes at the same depth |
| **leaf** | a node with no children |
| **degenerate** | every node has at most one child — a linked list wearing a tree's type |

The **−1 for empty** looks arbitrary and is the choice that makes `height(n) = 1 + max(height(left),
height(right))` work with no special case for a leaf: a leaf's children are both empty, so
$1 + \max(-1, -1) = 0$. Choose 0 for empty instead and every leaf needs its own branch. This is
NB-05 §1.3's sentinel argument again — pick the convention that deletes the special case.

**Why the height matters more than anything else in this notebook:** a recursive traversal's stack
depth *is* the tree's height. A balanced tree of $n$ nodes has height $\Theta(\log n)$; a
degenerate one has height $n - 1$. Same node count, same $\Theta(n)$ traversal time, and a
difference between "works on a million nodes" and "crashes on a thousand" — which §3 measures.

In [2]:
# ---------------------------------------------------------------------------
# The node, and the two tree shapes that matter.
# ---------------------------------------------------------------------------
class TNode:
    __slots__ = ("val", "left", "right")

    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_balanced(n):
    """A perfectly balanced tree of n nodes holding 0..n-1, in sorted order."""
    def go(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        return TNode(mid, go(lo, mid - 1), go(mid + 1, hi))
    return go(0, n - 1)


def build_degenerate(n):
    """A tree that is really a linked list: every node has only a left child."""
    if n == 0:
        return None
    root = TNode(0)
    cur = root
    for v in range(1, n):
        cur.left = TNode(v)
        cur = cur.left
    return root


def build_from_spec(spec):
    """A deterministic pseudo-random tree, for the stress tests.

    Each entry is a list of turns (0 = left, 1 = right) followed from the root;
    the new node is attached at the first free slot reached.
    """
    if not spec:
        return None
    root = TNode(0)
    for v, choices in enumerate(spec, start=1):
        cur = root
        placed = False
        for c in choices:
            if c == 0:
                if cur.left is None:
                    cur.left = TNode(v)
                    placed = True
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = TNode(v)
                    placed = True
                    break
                cur = cur.right
        if not placed:                     # ran out of turns: take whatever is free
            if cur.left is None:
                cur.left = TNode(v)
            elif cur.right is None:
                cur.right = TNode(v)
    return root


def height(node):
    """Edges to the deepest leaf. Empty tree is -1, which removes the leaf case."""
    if node is None:
        return -1
    return 1 + max(height(node.left), height(node.right))


def count_nodes(node):
    return 0 if node is None else 1 + count_nodes(node.left) + count_nodes(node.right)


print("  %-14s %10s %12s %16s" % ("shape", "nodes", "height", "height / log2(n)"))
print("  " + "-" * 56)
for n in (15, 1_023, 65_535):
    t = build_balanced(n)
    h = height(t)
    print("  %-14s %10s %12d %16.2f"
          % ("balanced", "{:,}".format(n), h, h / math.log2(n)))
for n in (15, 500, 1_000):
    t = build_degenerate(n)
    try:
        h = "%d" % height(t)
    except RecursionError:
        h = "RecursionError"
    print("  %-14s %10s %12s %16s" % ("degenerate", "{:,}".format(n), h, "n - 1"))

print()
print("Two things in that table, and the second one is not a typo.")
print()
print("1. A balanced tree's height is about log2(n): 15 nodes -> height 3,")
print("   65,535 nodes -> height 15. A degenerate tree's height is n - 1.")
print()
print("2. `height` is itself recursive, so measuring the height of a")
print("   1,000-node degenerate tree ALREADY exceeds Python's recursion")
print("   limit. The constraint Part 3 is about turned up while we were")
print("   merely trying to describe the tree.")

  shape               nodes       height height / log2(n)
  --------------------------------------------------------
  balanced               15            3             0.77
  balanced            1,023            9             0.90
  balanced           65,535           15             0.94
  degenerate             15           14            n - 1
  degenerate            500          499            n - 1
  degenerate          1,000 RecursionError            n - 1

Two things in that table, and the second one is not a typo.

1. A balanced tree's height is about log2(n): 15 nodes -> height 3,
   65,535 nodes -> height 15. A degenerate tree's height is n - 1.

2. `height` is itself recursive, so measuring the height of a
   1,000-node degenerate tree ALREADY exceeds Python's recursion
   limit. The constraint Part 3 is about turned up while we were
   merely trying to describe the tree.


## 1.2 The four traversals

Three of them differ by **one line's position**. That is not a mnemonic, it is literally the
difference:

```python
def preorder(n):   visit(n); go(n.left); go(n.right)     # node first
def inorder(n):    go(n.left); visit(n); go(n.right)     # node between
def postorder(n):  go(n.left); go(n.right); visit(n)     # node last
```

The names describe **where the node is visited relative to its subtrees**, and each answers a
different question:

| Traversal | Order | Use it when |
|---|---|---|
| **preorder** | node, left, right | you need the parent before its children — copying a tree, serialising (§2.4), rendering a document outline |
| **inorder** | left, node, right | the tree is a BST and you want **sorted** output (NB-07) |
| **postorder** | left, right, node | children must be finished first — freeing memory, computing subtree sizes or heights (§1.1's `height` is a postorder), evaluating an expression tree |
| **level order** | by depth | you want *nearest first* — shortest path in an unweighted graph, printing by level, §2.3's view problems |

**Level order is the odd one out, and it is the one worth pausing on.** It is not a recursion. You
cannot express "all of depth 1, then all of depth 2" naturally with the call stack, because the
call stack is a *stack* and it will always drive you down before across. Level order needs a
**queue** — which is NB-05 §1.1's "stack for depth, queue for breadth" stated as an algorithm
rather than a slogan. §1.3 makes the equivalence exact.

In [3]:
# ---------------------------------------------------------------------------
# 1.2 The four traversals, recursively (and level order, which cannot be).
# ---------------------------------------------------------------------------
def preorder(root):
    out = []

    def go(n):
        if n is None:
            return
        out.append(n.val)          # visit BEFORE the subtrees
        go(n.left)
        go(n.right)

    go(root)
    return out


def inorder(root):
    out = []

    def go(n):
        if n is None:
            return
        go(n.left)
        out.append(n.val)          # visit BETWEEN the subtrees
        go(n.right)

    go(root)
    return out


def postorder(root):
    out = []

    def go(n):
        if n is None:
            return
        go(n.left)
        go(n.right)
        out.append(n.val)          # visit AFTER the subtrees

    go(root)
    return out


def level_order(root):
    """Breadth-first. A QUEUE, not recursion."""
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        n = q.popleft()
        out.append(n.val)
        if n.left:
            q.append(n.left)
        if n.right:
            q.append(n.right)
    return out


#         1
#       /   \
#      2     3
#     / \
#    4   5
demo = TNode(1, TNode(2, TNode(4), TNode(5)), TNode(3))

print("        1")
print("      /   \\")
print("     2     3")
print("    / \\")
print("   4   5")
print()
print("  preorder    (node, left, right) ->", preorder(demo))
print("  inorder     (left, node, right) ->", inorder(demo))
print("  postorder   (left, right, node) ->", postorder(demo))
print("  level order (by depth)          ->", level_order(demo))
print()
print("  The first three visit the same nodes with one line moved.")
print("  The fourth is a different algorithm entirely.")

        1
      /   \
     2     3
    / \
   4   5

  preorder    (node, left, right) -> [1, 2, 4, 5, 3]
  inorder     (left, node, right) -> [4, 2, 5, 1, 3]
  postorder   (left, right, node) -> [4, 5, 2, 3, 1]
  level order (by depth)          -> [1, 2, 3, 4, 5]

  The first three visit the same nodes with one line moved.
  The fourth is a different algorithm entirely.


## 1.3 Converting recursion to an explicit stack

A recursive traversal already uses a stack — **the call stack** — it is just managed by the runtime
rather than by you. Converting to an explicit stack is therefore not inventing a new algorithm; it
is taking over the bookkeeping, which is exactly what you must do when the runtime's stack is too
small (§3).

**Preorder is the easy one**, because the visit happens before either recursive call, so there is
nothing to remember. Push the root; pop, visit, push the children. The only subtlety is **push
right first**, so left is popped first and the order comes out right — a stack reverses whatever
you put in.

**Inorder is harder**, because you must descend all the way left *before* visiting anything, and
then remember every node you passed. So the stack holds the path you came down.

**Postorder is the hard one**, because a node must be visited only after *both* subtrees, so
reaching a node from the stack is not enough — you need to know whether you are arriving for the
first time or returning from the right subtree. Two solutions:

1. **The trick:** postorder is `left, right, node`. Run a *mirrored preorder* — `node, right, left`
   — and reverse the result. Six lines, obviously correct once seen, and it costs an extra pass and
   $\Theta(n)$ output space.
2. **The honest one:** one stack plus a `last_visited` pointer, so you can tell "descending" from
   "returning". More code, no extra pass.

Both are below, and both are checked against the recursive version.

**And the punchline for level order:** the breadth-first traversal is the same six lines with the
queue swapped for a stack. That single substitution turns BFS into DFS, which is the clearest
possible demonstration that these containers are *policies* rather than storage.

In [4]:
# ---------------------------------------------------------------------------
# 1.3 The same traversals, with the stack made explicit.
# ---------------------------------------------------------------------------
def preorder_iter(root):
    if root is None:
        return []
    out, stack = [], [root]
    while stack:
        n = stack.pop()
        out.append(n.val)
        if n.right:
            stack.append(n.right)     # right first, so left pops first
        if n.left:
            stack.append(n.left)
    return out


def inorder_iter(root):
    """The stack holds the path down the left spine."""
    out, stack, cur = [], [], root
    while stack or cur is not None:
        while cur is not None:        # descend as far left as possible
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()             # nothing further left: visit
        out.append(cur.val)
        cur = cur.right               # then do the right subtree
    return out


def postorder_iter_reversed(root):
    """Mirrored preorder (node, right, left), reversed."""
    if root is None:
        return []
    out, stack = [], [root]
    while stack:
        n = stack.pop()
        out.append(n.val)
        if n.left:
            stack.append(n.left)      # left first this time
        if n.right:
            stack.append(n.right)
    return out[::-1]                  # node,right,left reversed == left,right,node


def postorder_iter_one_pass(root):
    """One stack plus a last-visited pointer, so we know why we are at a node."""
    out, stack, cur, last = [], [], root, None
    while stack or cur is not None:
        while cur is not None:
            stack.append(cur)
            cur = cur.left
        peek = stack[-1]
        if peek.right is not None and last is not peek.right:
            cur = peek.right          # right subtree not done yet: go there
        else:
            out.append(peek.val)      # both subtrees done: visit
            last = stack.pop()
    return out


def traverse_with(root, container_pop):
    """BFS or DFS depending only on which end we take from."""
    if root is None:
        return []
    out, pending = [], deque([root])
    while pending:
        n = container_pop(pending)
        out.append(n.val)
        if n.left:
            pending.append(n.left)
        if n.right:
            pending.append(n.right)
    return out


print("Same function, same data, one difference -- which end we remove from:")
print("  take from the FRONT (queue) ->", traverse_with(demo, lambda d: d.popleft()))
print("  take from the BACK  (stack) ->", traverse_with(demo, lambda d: d.pop()))
print("  recursive preorder          ->", preorder(demo))
print()
print("  The first is breadth-first, the second depth-first: NB-05 section 1.1's")
print("  'stack for depth, queue for breadth', as a one-character change.")
print()
print("  Note the stack version is not identical to preorder. It appends left")
print("  then right, and a stack hands them back in the opposite order, so it")
print("  descends RIGHT first. It is a genuine depth-first traversal, mirrored.")
print("  preorder_iter above pushes right first precisely to cancel that out --")
print("  which is the 'a stack reverses' point, visible in one line.")

Same function, same data, one difference -- which end we remove from:
  take from the FRONT (queue) -> [1, 2, 3, 4, 5]
  take from the BACK  (stack) -> [1, 3, 2, 5, 4]
  recursive preorder          -> [1, 2, 4, 5, 3]

  The first is breadth-first, the second depth-first: NB-05 section 1.1's
  'stack for depth, queue for breadth', as a one-character change.

  Note the stack version is not identical to preorder. It appends left
  then right, and a stack hands them back in the opposite order, so it
  descends RIGHT first. It is a genuine depth-first traversal, mirrored.
  preorder_iter above pushes right first precisely to cancel that out --
  which is the 'a stack reverses' point, visible in one line.


In [5]:
# ---------------------------------------------------------------------------
# Every iterative version, checked against its recursive twin.
# ---------------------------------------------------------------------------
def gen_spec(rng):
    """A deterministic description of a random tree."""
    n = rng.randrange(0, 22)
    return [tuple(rng.randrange(2) for _ in range(rng.randrange(0, 6))) for _ in range(n)]


PAIRS = (("preorder_iter", preorder_iter, preorder),
         ("inorder_iter", inorder_iter, inorder),
         ("postorder_iter_reversed", postorder_iter_reversed, postorder),
         ("postorder_iter_one_pass", postorder_iter_one_pass, postorder))

for name, impl, ref in PAIRS:
    checked = stress(lambda s, f=impl: f(build_from_spec(s)),
                     lambda s, g=ref: g(build_from_spec(s)),
                     gen_spec, n=3000, seed=RANDOM_SEED, label=name)
    print("%-26s %s random trees agree with the recursive version"
          % (name + ":", "{:,}".format(checked)))

print()
print("  On the demo tree, every pair matches:")
print("    preorder  rec %s  iter %s" % (preorder(demo), preorder_iter(demo)))
print("    inorder   rec %s  iter %s" % (inorder(demo), inorder_iter(demo)))
print("    postorder rec %s  iter %s" % (postorder(demo), postorder_iter_one_pass(demo)))

preorder_iter:             3,000 random trees agree with the recursive version


inorder_iter:              3,000 random trees agree with the recursive version


postorder_iter_reversed:   3,000 random trees agree with the recursive version
postorder_iter_one_pass:   3,000 random trees agree with the recursive version

  On the demo tree, every pair matches:
    preorder  rec [1, 2, 4, 5, 3]  iter [1, 2, 4, 5, 3]
    inorder   rec [4, 2, 5, 1, 3]  iter [4, 2, 5, 1, 3]
    postorder rec [4, 5, 2, 3, 1]  iter [4, 5, 2, 3, 1]


All four iterative versions agree with their recursive twins on 3,000 randomised trees each,
including empty and single-node trees.

Two things worth extracting, because they generalise past trees:

- **A stack reverses.** `preorder_iter` pushes right before left *because* the stack will hand them
  back in the opposite order. Every explicit-stack conversion has one of these reversals in it, and
  it is the most common place to get the order wrong. If your traversal is mirrored, this is why.
- **The hard part of postorder is knowing why you are at a node.** The recursion knew implicitly —
  the program counter was at a different point in the function. An explicit stack throws that away,
  so you have to store it, either as a `last_visited` pointer or by restructuring the problem
  (`postorder_iter_reversed`). **That is the general cost of converting recursion to iteration:
  the call stack was storing your position in the code, not just your data.**

## 1.4 Morris traversal — $O(1)$ space

Every traversal so far uses $\Theta(h)$ space, where $h$ is the height — the call stack or the
explicit one. On a degenerate tree that is $\Theta(n)$, which is what §3 is about.

**Morris traversal does it in $\Theta(1)$ space**, and the trick is audacious: it *modifies the
tree* while traversing, then puts it back.

**The idea.** In an inorder walk, after finishing a node's left subtree you must return to that
node — and that is the only reason you need a stack. But the last node visited in a left subtree is
its **rightmost** node (the inorder **predecessor**), and that node's `right` pointer is empty. So
temporarily point it at the current node. When the walk later arrives back along that thread, it
knows the left subtree is done, removes the thread, and visits.

For each node with a left child:

1. find its inorder predecessor — go left once, then right as far as possible;
2. if the predecessor's `right` is empty, **thread** it to the current node and descend left;
3. if it already points back at the current node, the left subtree is finished — **remove the
   thread**, visit, and go right.

**The property that makes this usable is that step 3 restores the tree.** Without it you have a
traversal that silently corrupts its input, which is far worse than one that uses memory. So that
is what we check: not just that the output is right, but that the tree is **structurally identical
afterwards**.

In [6]:
# ---------------------------------------------------------------------------
# 1.4 Morris inorder traversal.
# ---------------------------------------------------------------------------
def morris_inorder(root):
    """Inorder in O(1) space, by threading and unthreading the tree."""
    out, cur = [], root
    while cur is not None:
        if cur.left is None:
            out.append(cur.val)                     # nothing to the left: visit
            cur = cur.right
        else:
            pred = cur.left                          # find the inorder predecessor
            while pred.right is not None and pred.right is not cur:
                pred = pred.right
            if pred.right is None:
                pred.right = cur                     # thread, then descend left
                cur = cur.left
            else:
                pred.right = None                    # UNTHREAD: restore the tree
                out.append(cur.val)                  # left subtree finished: visit
                cur = cur.right
    return out


def tree_fingerprint(node):
    """A full structural description, for proving the tree came back unchanged."""
    if node is None:
        return None
    return (node.val, tree_fingerprint(node.left), tree_fingerprint(node.right))


checked = stress(lambda s: morris_inorder(build_from_spec(s)),
                 lambda s: inorder(build_from_spec(s)),
                 gen_spec, n=3000, seed=RANDOM_SEED, label="morris_inorder")
print("morris_inorder: %s random trees produce the same order as recursive inorder."
      % "{:,}".format(checked))


def morris_restores_tree(spec):
    """Traverse, then compare the tree with what it was before."""
    tree = build_from_spec(spec)
    before = tree_fingerprint(tree)
    morris_inorder(tree)
    return tree_fingerprint(tree) == before


checked = stress(morris_restores_tree, lambda s: True, gen_spec,
                 n=3000, seed=RANDOM_SEED, label="morris restores the tree")
print("                %s random trees are STRUCTURALLY IDENTICAL afterwards --"
      % "{:,}".format(checked))
print("                every thread it created was removed again.")

print()
print("  And the space, on the tree that costs the others the most:")
deep = build_degenerate(50_000)
print("    a 50,000-node degenerate tree (height 49,999):")
print("      morris_inorder ->", len(morris_inorder(deep)), "values, using O(1) extra space")
print("      inorder_iter   ->", len(inorder_iter(deep)),
      "values, but its stack held 50,000 nodes")
del deep

morris_inorder: 3,000 random trees produce the same order as recursive inorder.


                3,000 random trees are STRUCTURALLY IDENTICAL afterwards --
                every thread it created was removed again.

  And the space, on the tree that costs the others the most:
    a 50,000-node degenerate tree (height 49,999):
      morris_inorder -> 50000 values, using O(1) extra space
      inorder_iter   -> 50000 values, but its stack held 50,000 nodes


The output is right **and the tree comes back unchanged**, 3,000 times. That second check is the
one that matters: a traversal which corrupts its input is not a traversal, and "it restores the
tree" is a claim you should never take on faith when the algorithm's whole method is vandalism.

**What Morris costs**, because $O(1)$ space is not free:

- **Time is still $\Theta(n)$, but with a worse constant.** Finding each predecessor walks a right
  spine, so some edges are traversed up to three times. The amortised argument is the same shape as
  NB-05 §3.2's: each edge is threaded at most once and unthreaded at most once.
- **It mutates the tree mid-traversal.** So it is unusable if anything else can observe the tree
  concurrently, and it is impossible on an immutable tree. Abandoning the traversal part-way —
  an exception, a `break`, a generator that is never exhausted — **leaves threads in place** and
  the tree genuinely corrupted. That is a serious caveat and the reason it is rarely used in
  production.
- **It needs writable `right` pointers.** A tree of read-only nodes cannot be walked this way.

**When it is worth it:** embedded work with a hard memory budget, or traversing a tree so
degenerate that $\Theta(h)$ space is $\Theta(n)$ and $n$ is enormous. Otherwise the explicit stack
of §1.3 is the right answer — and §3 shows it is already enough to fix the problem Morris solves
more aggressively.

## 1.5 The same traversals in Java, cross-checked

Same tree, same four traversals, both languages run on identical randomly generated trees and
required to produce identical output.

The tree is described by a **serialised preorder with explicit nulls** — the format §2.4 derives —
so the two languages build provably the same structure rather than two structures we hope match.

In [7]:
# ---------------------------------------------------------------------------
# Python and Java traversing the same randomly generated trees.
# ---------------------------------------------------------------------------
JAVA_TREE_SRC = r"""
import java.util.*;

public class Traversals {
    static final class Node {
        int val; Node left, right;
        Node(int v) { val = v; }
    }

    static int pos;
    static String[] toks;

    /** Rebuild from preorder-with-nulls, exactly as Python serialised it. */
    static Node parse() {
        String t = toks[pos++];
        if (t.equals("#")) return null;
        Node n = new Node(Integer.parseInt(t));
        n.left = parse();
        n.right = parse();
        return n;
    }

    static void pre(Node n, StringBuilder sb) {
        if (n == null) return;
        sb.append(n.val).append(' ');
        pre(n.left, sb); pre(n.right, sb);
    }
    static void in(Node n, StringBuilder sb) {
        if (n == null) return;
        in(n.left, sb); sb.append(n.val).append(' '); in(n.right, sb);
    }
    static void post(Node n, StringBuilder sb) {
        if (n == null) return;
        post(n.left, sb); post(n.right, sb); sb.append(n.val).append(' ');
    }
    static void level(Node root, StringBuilder sb) {
        if (root == null) return;
        Deque<Node> q = new ArrayDeque<>();
        q.addLast(root);
        while (!q.isEmpty()) {
            Node n = q.pollFirst();
            sb.append(n.val).append(' ');
            if (n.left != null) q.addLast(n.left);
            if (n.right != null) q.addLast(n.right);
        }
    }

    public static void main(String[] args) throws Exception {
        Scanner sc = new Scanner(System.in);
        toks = sc.nextLine().trim().split(",");
        pos = 0;
        Node root = parse();
        StringBuilder sb = new StringBuilder();
        pre(root, sb);   sb.append('|');
        in(root, sb);    sb.append('|');
        post(root, sb);  sb.append('|');
        level(root, sb);
        System.out.println(sb.toString().replaceAll(" \\|", "|").trim());
    }
}
"""


def serialise_preorder(node):
    """Preorder with '#' for empty -- the format section 2.4 arrives at."""
    out = []

    def go(n):
        if n is None:
            out.append("#")
            return
        out.append(str(n.val))
        go(n.left)
        go(n.right)

    go(node)
    return ",".join(out)


def all_four(spec):
    t = build_from_spec(spec)
    parts = [preorder(t), inorder(t), postorder(t), level_order(t)]
    return "|".join(" ".join(str(v) for v in p) for p in parts)


if JAVA:
    checked = cross_check(all_four, JAVA_TREE_SRC, gen_spec,
                          lambda s: serialise_preorder(build_from_spec(s)) + "\n",
                          n=50, seed=RANDOM_SEED, label="traversals")
    print("Python and Java agree on all four traversals of %d random trees." % checked)
    print("The trees are transferred as preorder-with-nulls, so both languages")
    print("provably build the same structure rather than two we hope match.")
else:
    print("JDK not available; the cross-check did not run.")

Python and Java agree on all four traversals of 50 random trees.
The trees are transferred as preorder-with-nulls, so both languages
provably build the same structure rather than two we hope match.


Fifty randomly shaped trees, four traversals each, identical output from both languages.

The interesting part is the transfer format. Sending a tree between two programs means
**serialising** it, and the format used here — preorder with an explicit marker for empty — is
exactly the one §2.4 derives from first principles, for exactly this reason: it is the only one of
the four traversals that can be rebuilt unambiguously in a single pass.

Java's recursive traversals here are the same shape as Python's, and they have the same weakness.
§3 measures how much deeper Java can go before it hits the same wall.

***
# Part 2 - Worked problems

Four problems, each teaching a pattern that recurs far beyond trees.

| # | Problem | The pattern |
|---|---|---|
| 2.1 | Height and depth | choosing the convention that deletes the special case |
| 2.2 | Diameter | **return one value, track another** |
| 2.3 | Level-order variants | BFS with the level boundary made explicit |
| 2.4 | Serialise and deserialise | why only *one* traversal round-trips |

## 2.1 Height and depth, and the off-by-one everyone hits

§1.1 defined **height** as edges to the deepest leaf, with the empty tree at **−1**. There is a
competing convention counting *nodes* rather than edges, which puts a leaf at 1 and the empty tree
at 0. Neither is wrong; mixing them is, and mixing them is easy because both produce plausible
numbers on small trees.

**This notebook counts edges.** The rule for deciding is worth stating because it generalises: pick
the convention under which the recurrence has **no special case**.

```python
height(None) = -1
height(node) = 1 + max(height(left), height(right))
```

A leaf's children are both empty, so $1 + \max(-1, -1) = 0$ — correct, with no branch for leaves.
Choose 0 for the empty tree and every leaf needs its own case. This is NB-05 §1.3's sentinel
argument and NB-03 §2.3's `{0: 1}` seed: **design the special case away rather than handling it.**

The reference below computes the height by counting BFS levels — a completely different method — so
the two can only agree if the convention is applied consistently.

In [8]:
# ---------------------------------------------------------------------------
# 2.1 Height, two ways.
# ---------------------------------------------------------------------------
def height_bfs(root):
    """Count levels breadth-first. Independent of the recursive definition."""
    if root is None:
        return -1
    levels, q = -1, deque([root])
    while q:
        levels += 1
        for _ in range(len(q)):          # one full level per outer iteration
            n = q.popleft()
            if n.left:
                q.append(n.left)
            if n.right:
                q.append(n.right)
    return levels


checked = stress(lambda s: height(build_from_spec(s)),
                 lambda s: height_bfs(build_from_spec(s)),
                 gen_spec, n=3000, seed=RANDOM_SEED, label="height")
print("height: %s random trees -- the recursive definition and the BFS level"
      % "{:,}".format(checked))
print("        count agree exactly, including the empty tree.")
print()
print("  %-28s %8s" % ("tree", "height"))
print("  " + "-" * 38)
for label, t in (("empty", None),
                 ("single node", TNode(0)),
                 ("the demo tree", demo),
                 ("balanced, 1,023 nodes", build_balanced(1_023)),
                 ("degenerate, 500 nodes", build_degenerate(500))):
    print("  %-28s %8d" % (label, height(t)))

print()
print("  Note the empty tree is -1 and a single node is 0. If those look wrong,")
print("  you are using the node-counting convention -- which is fine, as long")
print("  as you never mix the two in one codebase.")

height: 3,000 random trees -- the recursive definition and the BFS level
        count agree exactly, including the empty tree.

  tree                           height
  --------------------------------------
  empty                              -1
  single node                         0
  the demo tree                       2
  balanced, 1,023 nodes               9
  degenerate, 500 nodes             499

  Note the empty tree is -1 and a single node is 0. If those look wrong,
  you are using the node-counting convention -- which is fine, as long
  as you never mix the two in one codebase.


## 2.2 Diameter — return one value, track another

**The problem.** The **diameter** is the number of edges on the longest path between any two nodes.
That path need not pass through the root.

**Why it is interesting.** The obvious recursion does not work. "The diameter of a tree is the
larger of its subtrees' diameters" is false — the longest path might cross the root, joining the
deepest point on the left to the deepest point on the right, and neither subtree's diameter sees
that.

**The pattern.** At each node, two different quantities are in play:

- what the **parent needs**: this subtree's *height*, since the parent can only extend a path
  downward;
- what the **answer needs**: the longest path *through* this node, which is
  `height(left) + height(right) + 2` edges.

So the recursion **returns the height and records the best diameter in a side channel.** That is
the pattern, and it is one of the most reusable ideas in tree work: *return what your caller needs;
accumulate what the problem needs.* The same shape solves maximum path sum, counting good nodes,
and most "hard" tree problems — they are hard exactly to the extent that people try to make the
return value do both jobs.

The reference is deliberately unrelated: treat the tree as an undirected graph and BFS from every
node. Slow, obviously correct, and shares no logic with the implementation.

In [9]:
# ---------------------------------------------------------------------------
# 2.2 Diameter: the recursion returns height, and records the answer separately.
# ---------------------------------------------------------------------------
def diameter(root):
    """Longest path between any two nodes, in edges."""
    best = 0

    def go(node):
        nonlocal best
        if node is None:
            return -1                                  # height convention from 2.1
        lh = go(node.left)
        rh = go(node.right)
        best = max(best, lh + rh + 2)                  # path THROUGH this node
        return 1 + max(lh, rh)                         # what the PARENT needs

    go(root)
    return max(best, 0)                                # empty/single tree: 0 edges


def diameter_brute(root):
    """Reference: treat it as an undirected graph, BFS from every node."""
    if root is None:
        return 0
    nodes, adj = [], {}

    def collect(n):
        if n is None:
            return
        nodes.append(n)
        adj.setdefault(id(n), [])
        for child in (n.left, n.right):
            if child is not None:
                adj[id(n)].append(child)
                adj.setdefault(id(child), []).append(n)
                collect(child)

    collect(root)
    best = 0
    for start in nodes:
        seen, q = {id(start): 0}, deque([start])
        while q:
            x = q.popleft()
            for y in adj.get(id(x), []):
                if id(y) not in seen:
                    seen[id(y)] = seen[id(x)] + 1
                    best = max(best, seen[id(y)])
                    q.append(y)
    return best


checked = stress(lambda s: diameter(build_from_spec(s)),
                 lambda s: diameter_brute(build_from_spec(s)),
                 gen_spec, n=3000, seed=RANDOM_SEED, label="diameter")
print("diameter: %s random trees agree with an all-pairs BFS reference"
      % "{:,}".format(checked))
print("          that shares no logic with the implementation.")
print()
print("  %-26s %10s %10s" % ("tree", "height", "diameter"))
print("  " + "-" * 48)
for label, t in (("empty", None), ("single node", TNode(0)),
                 ("the demo tree", demo),
                 ("balanced, 15 nodes", build_balanced(15)),
                 ("degenerate, 10 nodes", build_degenerate(10))):
    print("  %-26s %10d %10d" % (label, height(t), diameter(t)))

print()
print("  The demo tree's diameter is 3: the path 4-2-1-3 uses 3 edges and does")
print("  not run between two leaves of the same subtree. A recursion that only")
print("  combined subtree diameters would miss it.")

diameter: 3,000 random trees agree with an all-pairs BFS reference
          that shares no logic with the implementation.

  tree                           height   diameter
  ------------------------------------------------
  empty                              -1          0
  single node                         0          0
  the demo tree                       2          3
  balanced, 15 nodes                  3          6
  degenerate, 10 nodes                9          9

  The demo tree's diameter is 3: the path 4-2-1-3 uses 3 edges and does
  not run between two leaves of the same subtree. A recursion that only
  combined subtree diameters would miss it.


## 2.3 Level-order variants — making the level boundary explicit

§1.2's `level_order` returns one flat list, which loses the level boundaries. Most real questions
need them: print by level, take the last node of each level, alternate direction.

**The technique** is one line: before draining the queue, **record its current length**. That
number is exactly the size of the current level, because every node of the next level is enqueued
only while the current one is being processed.

```python
while q:
    for _ in range(len(q)):      # <- exactly this level, snapshotted
        ...
```

Get that wrong — iterate `while q` without the snapshot — and levels bleed into each other. It is a
small line carrying the entire structure of the algorithm, and once you have it, the variants are
trivial:

- **level order**: collect each level as a list;
- **zigzag**: reverse alternate levels;
- **right-side view**: keep the last node of each level;
- **bottom-up**: reverse the list of levels at the end.

In [10]:
# ---------------------------------------------------------------------------
# 2.3 BFS with the level boundary snapshotted.
# ---------------------------------------------------------------------------
def levels(root):
    """Level order, grouped by level."""
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):          # snapshot: exactly the current level
            n = q.popleft()
            level.append(n.val)
            if n.left:
                q.append(n.left)
            if n.right:
                q.append(n.right)
        out.append(level)
    return out


def zigzag(root):
    """Level order, reversing every other level."""
    out = levels(root)
    return [lv if i % 2 == 0 else lv[::-1] for i, lv in enumerate(out)]


def right_side_view(root):
    """What you would see standing to the right of the tree."""
    return [lv[-1] for lv in levels(root)]


def levels_reference(root):
    """Reference: group by computed depth, using a completely different method."""
    by_depth = {}

    def go(n, d):
        if n is None:
            return
        by_depth.setdefault(d, []).append(n.val)
        go(n.left, d + 1)
        go(n.right, d + 1)

    go(root, 0)
    return [by_depth[d] for d in sorted(by_depth)]


checked = stress(lambda s: levels(build_from_spec(s)),
                 lambda s: levels_reference(build_from_spec(s)),
                 gen_spec, n=3000, seed=RANDOM_SEED, label="levels")
print("levels: %s random trees match a reference that groups by computed depth"
      % "{:,}".format(checked))
print("        rather than by queue snapshots -- two unrelated methods agreeing.")

print()
t = build_balanced(9)
print("  a balanced tree of 9 nodes:")
print("    levels           ", levels(t))
print("    zigzag           ", zigzag(t))
print("    right-side view  ", right_side_view(t))
print("    bottom-up        ", levels(t)[::-1])

levels: 3,000 random trees match a reference that groups by computed depth
        rather than by queue snapshots -- two unrelated methods agreeing.

  a balanced tree of 9 nodes:
    levels            [[4], [1, 6], [0, 2, 5, 7], [3, 8]]
    zigzag            [[4], [6, 1], [0, 2, 5, 7], [8, 3]]
    right-side view   [4, 6, 7, 8]
    bottom-up         [[3, 8], [0, 2, 5, 7], [1, 6], [4]]


## 2.4 Serialise and deserialise — why only one traversal round-trips

**The problem.** Turn a tree into a string, and the string back into an identical tree.

**The question worth asking first:** which traversal can be rebuilt from? It is not obvious, and
the answer is genuinely restrictive.

- **Inorder alone is hopeless.** `[1, 2, 3]` is the inorder of *five* different shapes. Inorder
  tells you the left-to-right order and nothing about the structure.
- **Preorder alone is also ambiguous** — until you write down the empty subtrees. `1, 2` could be
  `2` as either child of `1`.
- **Preorder with explicit nulls is unambiguous**, and rebuilds in one pass, because the first
  token is always the root and the recursion consumes exactly its own subtree before returning.
  That is the format §1.5 already used to hand trees to Java.
- **Preorder + inorder together** also determine the tree uniquely without null markers, provided
  values are distinct — the classic reconstruction problem. Postorder + inorder likewise.
  **Preorder + postorder does not**, which surprises people: those two cannot distinguish a single
  left child from a single right child.

**The property to test is the round trip**, not the string. Whatever format you choose,
`deserialize(serialize(t))` must be structurally identical to `t` — so the test compares *trees*,
via the fingerprint from §1.4, rather than comparing strings and hoping.

In [11]:
# ---------------------------------------------------------------------------
# 2.4 Preorder with explicit nulls, and the round-trip property.
# ---------------------------------------------------------------------------
def serialize(root):
    """Preorder, with '#' marking an empty subtree."""
    out = []

    def go(n):
        if n is None:
            out.append("#")
            return
        out.append(str(n.val))
        go(n.left)
        go(n.right)

    go(root)
    return ",".join(out)


def deserialize(text):
    """Rebuild in one pass: the recursion consumes exactly its own subtree."""
    tokens = iter(text.split(","))

    def go():
        tok = next(tokens)
        if tok == "#":
            return None
        node = TNode(int(tok))
        node.left = go()
        node.right = go()
        return node

    return go()


checked = stress(lambda s: tree_fingerprint(deserialize(serialize(build_from_spec(s)))),
                 lambda s: tree_fingerprint(build_from_spec(s)),
                 gen_spec, n=3000, seed=RANDOM_SEED, label="serialize round-trip")
print("round trip: %s random trees survive serialize -> deserialize" % "{:,}".format(checked))
print("            STRUCTURALLY IDENTICAL -- compared as trees, not as strings.")

print()
print("  demo tree serialised:", serialize(demo))
print("  rebuilt inorder     :", inorder(deserialize(serialize(demo))))
print("  original inorder    :", inorder(demo))

print()
print("  And why inorder alone cannot work -- these are different trees:")
a = TNode(2, TNode(1), TNode(3))
b = TNode(1, None, TNode(2, None, TNode(3)))
c = TNode(3, TNode(1, None, TNode(2)), None)
for name, t in (("balanced", a), ("right spine", b), ("left-leaning", c)):
    print("    %-14s inorder %s   serialised %s"
          % (name, inorder(t), serialize(t)))
print("    Same inorder, three different trees. The nulls are what disambiguate.")

round trip: 3,000 random trees survive serialize -> deserialize
            STRUCTURALLY IDENTICAL -- compared as trees, not as strings.

  demo tree serialised: 1,2,4,#,#,5,#,#,3,#,#
  rebuilt inorder     : [4, 2, 5, 1, 3]
  original inorder    : [4, 2, 5, 1, 3]

  And why inorder alone cannot work -- these are different trees:
    balanced       inorder [1, 2, 3]   serialised 2,1,#,#,3,#,#
    right spine    inorder [1, 2, 3]   serialised 1,#,2,#,3,#,#
    left-leaning   inorder [1, 2, 3]   serialised 3,1,#,2,#,#,#
    Same inorder, three different trees. The nulls are what disambiguate.


***
# Part 3 - The signature difficulty: recursion depth is a real constraint

Trees make recursion natural, and §1.1 already showed what that costs: computing the height of a
1,000-node degenerate tree raised `RecursionError` before we had written a single algorithm.

This part measures the constraint properly, because the folk understanding of it is wrong in a
specific and important way. People say "recursion is risky for large inputs". The truth is
narrower and much more useful:

> **The limit is on depth, not size.** A recursive traversal's stack depth is the tree's *height*.

A million-node balanced tree has height ~20 and recurses without noticing. A thousand-node
degenerate tree has height 999 and dies. Same algorithm, same runtime, 1,000× difference in size,
opposite outcomes.

## 3.1 Where it breaks, and where it does not

In [12]:
# ---------------------------------------------------------------------------
# 3.1 The same recursive traversal on two tree shapes.
# ---------------------------------------------------------------------------
def try_traversal(fn, tree):
    try:
        return "ok (%s nodes)" % "{:,}".format(len(fn(tree)))
    except RecursionError:
        return "RecursionError"


print("Python's recursion limit: %d frames" % sys.getrecursionlimit())
print()
print("DEGENERATE trees -- height = n - 1:")
print("  %10s %10s %22s %20s" % ("nodes", "height", "recursive inorder", "iterative inorder"))
print("  " + "-" * 66)
for n in (100, 500, 900, 1_000, 10_000):
    t = build_degenerate(n)
    print("  %10s %10s %22s %20s"
          % ("{:,}".format(n), "{:,}".format(n - 1),
             try_traversal(inorder, t), try_traversal(inorder_iter, t)))
    del t

print()
print("BALANCED trees -- height = log2(n):")
print("  %10s %10s %22s %20s" % ("nodes", "height", "recursive inorder", "iterative inorder"))
print("  " + "-" * 66)
for n in (1_000, 100_000, 1_000_000):
    t = build_balanced(n)
    print("  %10s %10s %22s %20s"
          % ("{:,}".format(n), height(t),
             try_traversal(inorder, t), try_traversal(inorder_iter, t)))
    del t

print()
print("A 1,000,000-node balanced tree recurses fine. A 1,000-node degenerate")
print("one does not. The tree that is a THOUSAND TIMES SMALLER is the one that")
print("crashes, because depth is what costs stack frames -- not node count.")

Python's recursion limit: 1000 frames

DEGENERATE trees -- height = n - 1:
       nodes     height      recursive inorder    iterative inorder
  ------------------------------------------------------------------
         100         99         ok (100 nodes)       ok (100 nodes)
         500        499         ok (500 nodes)       ok (500 nodes)
         900        899         ok (900 nodes)       ok (900 nodes)
       1,000        999         RecursionError     ok (1,000 nodes)
      10,000      9,999         RecursionError    ok (10,000 nodes)

BALANCED trees -- height = log2(n):
       nodes     height      recursive inorder    iterative inorder
  ------------------------------------------------------------------
       1,000          9       ok (1,000 nodes)     ok (1,000 nodes)


     100,000         16     ok (100,000 nodes)   ok (100,000 nodes)


   1,000,000         19   ok (1,000,000 nodes) ok (1,000,000 nodes)

A 1,000,000-node balanced tree recurses fine. A 1,000-node degenerate
one does not. The tree that is a THOUSAND TIMES SMALLER is the one that
crashes, because depth is what costs stack frames -- not node count.


**The 1,000-node degenerate tree crashes; the 1,000,000-node balanced tree does not.**

That is the whole lesson, and it reframes the usual advice. "Avoid recursion on large inputs" is
the wrong rule — it would have you rewrite the million-node traversal, which is fine, and would not
warn you about the thousand-node one, which is not. The right rule:

> **Ask what bounds the depth.** If depth is $\Theta(\log n)$ — a *balanced* tree, binary search,
> merge sort's recursion tree — recursion is safe essentially forever, because $\log_2 10^9 \approx
> 30$. If depth can be $\Theta(n)$ — an unbalanced tree, a linked list (NB-04 §2.1), a graph path
> (NB-20) — it will break, and the only question is on what input.

And the uncomfortable part: **you often do not control the shape.** A BST built from sorted input
degenerates completely (NB-07), which is exactly why balanced trees exist (NB-08). A tree built
from user-supplied data is a tree whose height an attacker chooses — the same "who chooses the
input?" question as NB-02 §3, NB-03 §3 and NB-05 §3.3, arriving a fourth time.

## 3.2 The three fixes, and which to use

**Do not reach for `sys.setrecursionlimit` first.** It is the tempting one and the worst one: the
limit exists to raise a clean `RecursionError` *before* the interpreter's C stack overflows. Raise
it far enough and you replace a catchable Python exception with a **segfault**, which takes the
process down with no traceback.

The cell below demonstrates that the limit is a guard rail rather than the actual constraint — it
raises it just enough to make a specific tree work, which is safe, while noting why raising it
without bound is not.

In [13]:
# ---------------------------------------------------------------------------
# 3.2 Fix 1: raise the limit (carefully). Fix 2: iterate. Fix 3: Morris.
# ---------------------------------------------------------------------------
tree = build_degenerate(5_000)
original_limit = sys.getrecursionlimit()

print("A degenerate tree of 5,000 nodes.")
print()
print("  fix 0 -- do nothing:            ", try_traversal(inorder, tree))

sys.setrecursionlimit(20_000)                 # comfortably above the 5,000 needed
print("  fix 1 -- raise the limit:       ", try_traversal(inorder, tree))
sys.setrecursionlimit(original_limit)

print("  fix 2 -- explicit stack:        ", try_traversal(inorder_iter, tree))
print("  fix 3 -- Morris (O(1) space):   ", try_traversal(morris_inorder, tree))
print()
print("  recursion limit restored to", sys.getrecursionlimit())

print()
print("All three work here. They are not equally good:")
print()
print("  fix 1 trades a catchable RecursionError for an uncatchable crash if you")
print("        go too far -- the limit guards the C stack, which has its own")
print("        hard size. It is a stopgap, and it needs a number you cannot")
print("        always know in advance.")
print("  fix 2 is O(h) heap memory instead of O(h) stack, and the heap is far")
print("        larger. This is the normal answer.")
print("  fix 3 is O(1) space, and pays for it by mutating the tree (1.4).")
del tree

A degenerate tree of 5,000 nodes.

  fix 0 -- do nothing:             RecursionError
  fix 1 -- raise the limit:        ok (5,000 nodes)
  fix 2 -- explicit stack:         ok (5,000 nodes)
  fix 3 -- Morris (O(1) space):    ok (5,000 nodes)

  recursion limit restored to 1000

All three work here. They are not equally good:

  fix 1 trades a catchable RecursionError for an uncatchable crash if you
        go too far -- the limit guards the C stack, which has its own
        hard size. It is a stopgap, and it needs a number you cannot
        always know in advance.
  fix 2 is O(h) heap memory instead of O(h) stack, and the heap is far
        larger. This is the normal answer.
  fix 3 is O(1) space, and pays for it by mutating the tree (1.4).


In [14]:
# ---------------------------------------------------------------------------
# What the iterative version actually costs: heap instead of stack.
# ---------------------------------------------------------------------------
def max_stack_depth(root):
    """Instrumented inorder_iter: the largest the explicit stack ever gets."""
    stack, cur, peak = [], root, 0
    while stack or cur is not None:
        while cur is not None:
            stack.append(cur)
            peak = max(peak, len(stack))
            cur = cur.left
        cur = stack.pop()
        cur = cur.right
    return peak


print("  %-28s %10s %16s" % ("tree", "height", "peak stack size"))
print("  " + "-" * 58)
for label, t in (("balanced, 1,023 nodes", build_balanced(1_023)),
                 ("balanced, 65,535 nodes", build_balanced(65_535)),
                 ("degenerate, 10,000 nodes", build_degenerate(10_000))):
    h = height(t) if label.startswith("balanced") else 9_999
    print("  %-28s %10s %16s" % (label, "{:,}".format(h),
                                 "{:,}".format(max_stack_depth(t))))
    del t

print()
print("The explicit stack holds exactly height + 1 nodes at its peak -- the same")
print("quantity the call stack was holding. Nothing was made asymptotically")
print("cheaper; the storage simply moved from the C stack, which is a few")
print("megabytes, to the heap, which is gigabytes. That is the entire fix.")

  tree                             height  peak stack size
  ----------------------------------------------------------
  balanced, 1,023 nodes                 9               10
  balanced, 65,535 nodes               15               16
  degenerate, 10,000 nodes          9,999           10,000

The explicit stack holds exactly height + 1 nodes at its peak -- the same
quantity the call stack was holding. Nothing was made asymptotically
cheaper; the storage simply moved from the C stack, which is a few
megabytes, to the heap, which is gigabytes. That is the entire fix.


## 3.3 Java on the same axis

Java hits the same wall, at a different place, and with a different failure mode: **`StackOverflowError`**
rather than a Python exception, and it is an `Error` rather than an `Exception` — signalling that
you are not expected to catch it and carry on.

How deep can it go? That depends on the **thread's stack size**, which is settable — with `-Xss` on
the command line, or per-thread by constructing a `Thread` with an explicit size. The cell below
uses the per-thread form, because it is portable and shows the relationship directly.

In [15]:
# ---------------------------------------------------------------------------
# 3.3 How deep can a JVM thread recurse?
# ---------------------------------------------------------------------------
DEPTH_SRC = r"""
public class StackDepth {
    static int depth;
    static void recurse() { depth++; recurse(); }

    /** Recurse on a thread with the given stack size; report the depth reached. */
    static int probe(long stackBytes) throws InterruptedException {
        final int[] result = new int[1];
        Runnable r = () -> {
            depth = 0;
            try { recurse(); } catch (StackOverflowError e) { /* expected */ }
            result[0] = depth;
        };
        Thread t = new Thread(null, r, "probe", stackBytes);
        t.start();
        t.join();
        return result[0];
    }

    public static void main(String[] args) throws Exception {
        System.out.println("java.version = " + System.getProperty("java.version"));
        System.out.println();
        System.out.printf("  %-26s %16s%n", "thread stack size", "frames reached");
        System.out.println("  " + "-".repeat(44));
        System.out.printf("  %-26s %16s%n", "default", String.format("%,d", probe(0)));
        for (long mb : new long[]{1, 8, 64}) {
            System.out.printf("  %-26s %16s%n", mb + " MB",
                              String.format("%,d", probe(mb * 1024 * 1024)));
        }
    }
}
"""

if JAVA:
    print(run_java(DEPTH_SRC, timeout=600))
    print()
    print("For comparison, CPython's default limit is %d." % sys.getrecursionlimit())
else:
    print("JDK not available; skipping.")

java.version = 25.0.4.1

  thread stack size            frames reached
  --------------------------------------------
  default                              23,058
  1 MB                                 24,890
  8 MB                                208,436
  64 MB                             2,974,046


For comparison, CPython's default limit is 1000.


**A default JVM thread reaches roughly 23,000 frames against CPython's 1,000** — about 23× — and
the depth grows roughly in proportion to the stack the thread is given, into the millions at 64 MB.
(The exact counts move between runs; the orders of magnitude do not.)

Three things worth taking from that table:

- **Java's advantage is real but is a difference of degree.** 23,000 is a lot more than 1,000 and
  it is still nowhere near $n$ for any interesting $n$. A degenerate tree of 100,000 nodes kills
  both. The rule from §3.1 — *ask what bounds the depth* — is unchanged by having a bigger stack.
- **The number is a configuration, not a property of the language.** `-Xss64m` buys millions of
  frames. So "how deep can Java recurse?" has no answer without knowing the deployment, which is a
  bad thing to depend on: the code works on your laptop and overflows in a container with different
  defaults.
- **`StackOverflowError` is an `Error`.** Catching it and continuing is not safe in general — you
  do not know which frames were unwound or what invariants were left half-established. The cell
  above catches one deliberately, in a thread that does nothing else, which is about the only
  defensible use.

Python's `RecursionError` is friendlier — a normal exception, raised by a limit chosen to fire
*before* the real stack runs out — and that friendliness is exactly why raising the limit is
dangerous. The guard rail is the feature.

**The conclusion this notebook actually wants you to take away** is not "prefer iteration". It is:

> Recursion is the right tool for tree work, and its depth is the tree's height. When the height is
> bounded — a balanced tree, a search, a divide-and-conquer split — use it without hesitation.
> When the height is $\Theta(n)$ or attacker-controlled, convert to an explicit stack. The
> conversion is mechanical (§1.3), the cost is the same $\Theta(h)$ memory moved from a small stack
> to a large heap (§3.2), and NB-08's balanced trees exist precisely so the height is never the
> problem in the first place.

***
# Part 4 - Tough questions

***

### Q1. Name the four traversals and say when you would use each.

<details><summary>Answer</summary>

Three differ by **where one line goes**; the fourth is a different algorithm.

| Traversal | Order | Use it for |
|---|---|---|
| **preorder** | node, left, right | copying a tree, **serialising** (§2.4), rendering an outline, prefix expressions |
| **inorder** | left, node, right | **sorted output from a BST** (NB-07) — its defining application |
| **postorder** | left, right, node | anything where children must finish first: freeing memory, computing sizes or heights (§1.1's `height` *is* a postorder), evaluating an expression tree |
| **level order** | by depth | *nearest first* — shortest path in an unweighted graph, printing by level, §2.3's view problems |

**The distinction worth understanding**, not memorising: preorder decides about a node *before*
seeing its subtrees, postorder *after*. So any question of the form "what does this subtree
contain / cost / measure?" is postorder, and any question of the form "what context does this node
inherit?" is preorder.

**Level order is not a recursion.** You cannot naturally express "all of depth 1, then all of
depth 2" with the call stack, because a stack always drives you down before across. It needs a
**queue** — NB-05 §1.1's "stack for depth, queue for breadth", and §1.3 shows the two are the same
six lines with one container swapped.

</details>

***

### Q2. Convert a recursive traversal to an explicit stack.

<details><summary>Answer</summary>

The recursion already uses a stack — **the call stack**. You are taking over its bookkeeping, not
inventing an algorithm.

**Preorder is mechanical**, because the visit happens before both recursive calls, so there is
nothing to remember: push the root; pop, visit, push children. **Push right first** so left pops
first — a stack reverses whatever you give it, and this is the most common place to get the order
wrong.

**Inorder needs the path**: descend all the way left pushing as you go, pop and visit, then move
right. The stack holds the ancestors you still owe a visit to.

**Postorder is the hard one**, because arriving at a node from the stack does not tell you *why* —
first visit, or returning from the right subtree? Two answers:

1. **The trick:** run a mirrored preorder (node, right, left) and reverse the output. Six lines,
   obviously correct once seen, costs an extra pass.
2. **The honest one:** one stack plus a `last_visited` pointer to distinguish the cases.

**The general lesson**, which is what makes this question worth asking: *the call stack was storing
your position in the code, not just your data.* Preorder converts trivially because there is no
position to remember; postorder is painful because there is. Every recursion-to-iteration
conversion pays exactly that.

</details>

***

### Q3. Why is recursion natural for trees, and when does it break?

<details><summary>Answer</summary>

**Natural** because the data is *defined* recursively: a tree is empty, or a node with two trees.
Functions over recursively defined data are written by cases, and the base case is not something
you remember to add — it is half the definition.

**It breaks on depth, and the folk version of this is wrong.** People say "recursion is risky for
large inputs". §3.1 measured the truth:

| Tree | Nodes | Height | Recursive traversal |
|---|---|---|---|
| degenerate | **1,000** | 999 | **RecursionError** |
| balanced | **1,000,000** | 19 | **fine** |

**The tree a thousand times smaller is the one that crashes.** A recursive traversal's stack depth
is the tree's *height*, not its size.

So the rule is: **ask what bounds the depth.**

- $\Theta(\log n)$ — balanced tree, binary search, merge sort's recursion — safe essentially
  forever, since $\log_2 10^9 \approx 30$.
- $\Theta(n)$ — unbalanced tree, linked list (NB-04 §2.1), graph path (NB-20) — will break; the
  only question is on which input.

And the uncomfortable part: **you often do not choose the shape.** A BST built from sorted input is
degenerate (NB-07), which is why balanced trees exist (NB-08). A tree built from user data has a
height an attacker picks — the same "who chooses the input?" question as NB-02 §3, NB-03 §3 and
NB-05 §3.3.

</details>

***

### Q4. How do you fix a `RecursionError`? Rank the options.

<details><summary>Answer</summary>

**1. Explicit stack — the normal answer.** Mechanical (§1.3), and §3.2 measured what it actually
buys: the explicit stack holds `height + 1` nodes at its peak, *exactly what the call stack held*.
Nothing became asymptotically cheaper. The storage moved from the C stack — a few megabytes — to
the heap, which is gigabytes. That is the entire fix, and it is enough.

**2. Fix the tree, not the traversal.** If the height is the problem, a balanced tree (NB-08) makes
it $\Theta(\log n)$ and every recursive algorithm over it becomes safe at once. This is the real
answer when you control the structure.

**3. Morris traversal — $\Theta(1)$ space** (§1.4), by threading pointers through the tree and
removing them. Rarely worth it: it mutates the tree mid-traversal, so it is unusable concurrently,
impossible on an immutable tree, and **leaves the tree corrupted if you abandon the walk part-way**.

**4. `sys.setrecursionlimit` — last, and know why.** The limit exists to raise a clean
`RecursionError` *before* the interpreter's C stack overflows. Raise it far enough and you trade a
catchable exception for a **segfault** with no traceback. It also needs a number you often cannot
know in advance. Legitimate as a stopgap with a bounded, known depth; not a fix.

**The Java equivalents:** `-Xss` or a `Thread` constructed with an explicit stack size, which §3.3
measured — a default thread reaches ~23,000 frames, 64 MB reaches millions. Note the consequence:
"how deep can Java recurse?" has no answer without knowing the deployment, so code that works on
your laptop can overflow in a container with different defaults.

</details>

***

### Q5. Explain Morris traversal, and what it costs.

<details><summary>Answer</summary>

Inorder in $\Theta(1)$ space, by **temporarily rewiring the tree**.

The only reason inorder needs a stack is to get back to a node after finishing its left subtree.
But the last node visited in that subtree is its **rightmost** node — the inorder predecessor — and
that node's `right` pointer is empty. So point it at the current node. When the walk arrives back
along that thread, the left subtree is done: remove the thread, visit, go right.

For each node with a left child: find the predecessor (left once, then right as far as possible);
if its `right` is empty, **thread** and descend left; if it already points back, **unthread**,
visit, go right.

**The property that makes it usable is that it restores the tree**, and §1.4 checks exactly that —
not just that the output is right, but that the tree is **structurally identical afterwards**
across 3,000 randomised trees. A traversal whose method is vandalism should never be trusted on the
author's word.

**What it costs:**

- **Time is still $\Theta(n)$ with a worse constant** — finding predecessors re-walks right spines,
  so some edges are traversed up to three times. Each edge is threaded once and unthreaded once,
  which is the amortised argument of NB-05 §3.2.
- **It mutates during traversal** — unusable if anything else can observe the tree, impossible on
  an immutable one, and **abandoning the walk part-way leaves threads in place**, genuinely
  corrupting the tree. An exception, a `break`, or an unexhausted generator all do this.
- **It needs writable `right` pointers.**

**When it is worth it:** a hard memory budget, or a tree so degenerate that $\Theta(h)$ is
$\Theta(n)$ with huge $n$. Otherwise the explicit stack is right — §3.2 shows it already solves the
problem.

</details>

***

### Q6. Height or depth — which is which, and why does the empty tree matter?

<details><summary>Answer</summary>

**Depth** counts edges *down from the root* (root = 0). **Height** counts edges *down to the
deepest leaf* (leaf = 0). A tree's height is its root's height. They are measured from opposite
ends, which is why "the depth of the tree" is ambiguous and "the height of the tree" is not.

**The empty tree is −1** in this notebook, and the reason is worth generalising:

```python
height(None) = -1
height(node) = 1 + max(height(left), height(right))
```

A leaf's children are both empty, so $1 + \max(-1,-1) = 0$ — correct, **with no special case for
leaves**. Choose 0 for empty and every leaf needs its own branch.

The rule: **pick the convention under which the recurrence has no special case.** It is the same
move as NB-05 §1.3's sentinel node and NB-03 §2.3's `{0: 1}` prefix seed — *design the special case
away rather than handling it correctly.*

**The competing convention** counts nodes, putting a leaf at 1 and the empty tree at 0. Neither is
wrong; **mixing them is**, and mixing is easy because both give plausible numbers on small trees.
§2.1 verifies the recursive definition against an independent BFS level count, which only agrees if
the convention is applied consistently — that is the point of using an unrelated reference.

</details>

***

### Q7. Compute a tree's diameter. Why is the obvious recursion wrong?

<details><summary>Answer</summary>

**The obvious recursion is wrong** because "the diameter is the larger of the subtrees' diameters"
misses paths that *cross* the current node — joining the deepest point on the left to the deepest
point on the right. Neither subtree's diameter can see that path.

**The fix is the pattern worth learning.** At each node two different quantities are in play:

- what the **parent** needs: this subtree's **height**, since a parent can only extend a path
  downward;
- what the **answer** needs: the longest path *through* this node,
  `height(left) + height(right) + 2` edges.

So the recursion **returns the height and accumulates the diameter in a side channel**
(`nonlocal`, a member field, or a one-element list).

> **Return what your caller needs; accumulate what the problem needs.**

That shape solves maximum path sum, counting good nodes, longest univalue path, and most tree
problems labelled "hard" — they are hard precisely to the extent that people try to make one return
value do both jobs.

**Verification note:** §2.2's reference treats the tree as an undirected graph and BFSes from every
node — deliberately sharing no logic with the implementation, so agreement is evidence rather than
a coincidence of a shared misunderstanding.

</details>

***

### Q8. Serialise a tree so it can be rebuilt. Which traversals work?

<details><summary>Answer</summary>

Fewer than you would hope.

- **Inorder alone: hopeless.** `[1, 2, 3]` is the inorder of five different shapes — §2.4 prints
  three of them. Inorder gives left-to-right order and nothing about structure.
- **Preorder alone: ambiguous** — `1, 2` could be `2` as either child of `1`.
- **Preorder with explicit nulls: works**, and rebuilds in **one pass**, because the first token is
  always the root and each recursive call consumes exactly its own subtree before returning. This
  is the format §1.5 used to hand trees to Java.
- **Preorder + inorder: works** without null markers, if values are distinct — the classic
  reconstruction problem. **Postorder + inorder** likewise.
- **Preorder + postorder: does not work.** They cannot distinguish a single left child from a
  single right child. This surprises people and is worth checking on a two-node tree.

**The thing to test is the round trip, not the string.** §2.4 verifies
`deserialize(serialize(t))` is **structurally identical** to `t` via a fingerprint — comparing
*trees*, not comparing strings and hoping. Whatever format you pick, that is the property that
matters, and it is the one a string comparison can silently pass while the structure is wrong.

**In practice:** level order with nulls is what LeetCode prints and is friendlier to read; preorder
with nulls is easier to write and rebuild. Both are fine. What is not fine is a format that cannot
represent the empty subtree.

</details>

***

### Q9. BFS by level — how do you know where a level ends?

<details><summary>Answer</summary>

**Snapshot the queue's length before draining it.**

```python
while q:
    for _ in range(len(q)):     # exactly the current level
        ...
```

That works because every node of the next level is enqueued *only while the current level is being
processed*, so `len(q)` at the top of the outer loop is exactly the current level's size.

Get it wrong — iterate `while q` with no snapshot — and levels bleed together. It is one line
carrying the whole structure of the algorithm, and once you have it the variants are trivial
(§2.3): **zigzag** reverses alternate levels, **right-side view** keeps each level's last node,
**bottom-up** reverses the list of levels, **average per level** is a `sum` over each.

**The alternatives**, worth knowing so you can pick:

- **Sentinel node:** enqueue a `None` marker between levels. Works, needs care to avoid an infinite
  loop when re-enqueueing the marker.
- **Store `(node, depth)` pairs:** simplest to get right, and it lets you do a DFS instead — §2.3's
  reference groups by *computed depth* precisely to have an implementation sharing no logic with
  the queue-snapshot version.

**The point worth remembering:** level order is BFS, BFS is a queue, and on an unweighted graph BFS
gives shortest paths (NB-20). "Minimum depth of a tree" should be BFS, not DFS, because BFS can
stop at the first leaf it meets while DFS must explore everything.

</details>

***

### Q10. What are plain binary trees bad at?

<details><summary>Answer</summary>

**Everything requiring an ordering guarantee — because a plain binary tree has no invariant at
all.** It is a shape, not a data structure with promises. Consequences:

- **Search is $\Theta(n)$.** With no ordering, finding a value means visiting every node. That is
  no better than a linked list, which is why NB-07 adds the BST invariant — the ordering is what
  buys $O(h)$ search, and NB-08's balancing is what makes $h$ be $\log n$.
- **No sorted iteration.** Inorder gives sorted output *only* for a BST. On an arbitrary binary
  tree it gives an arbitrary order.
- **No balance guarantee**, which is §3's entire subject: nothing stops a tree being a linked list,
  and then every operation degrades and recursion crashes.
- **Poor locality.** Every node is a separate allocation, so traversal is pointer-chasing — NB-04
  §3.1 measured what that costs (about 30× against a contiguous array in Java). A heap (NB-09) is a
  tree stored *in an array* precisely to avoid this, and it is worth noticing that this is
  available whenever the tree is complete.
- **Memory per node.** Two pointers plus an object header per element. NB-04 §1.1 measured 56 bytes
  for a doubly linked node in Python against 8 for a list slot; a tree node is the same shape.

**What they are good for:** representing genuinely hierarchical data — expression trees, parse
trees, file systems, decision trees, the DOM — where the *structure is the information* rather than
an index into it. There, the shape is given by the problem and none of the above is a complaint.

</details>

***

### Q11. Iterative or recursive — how do you decide in real code?

<details><summary>Answer</summary>

**Default to recursive.** It matches the data definition (§1.1), it is shorter, and it is far
easier to get right — §1.3's postorder is the proof, where the iterative version needs a
`last_visited` pointer to recover information the recursion had for free.

**Switch to iterative when you can answer "yes" to any of these:**

1. **Can the depth be $\Theta(n)$?** Unbalanced trees, degenerate input, linked structures. §3.1's
   table is the argument: 1,000 nodes is enough.
2. **Is the shape attacker-controlled?** Then assume the worst height.
3. **Do you need to pause, resume, or abandon the traversal?** An explicit stack is a value you can
   store, serialise or step through; the call stack is not. This is why generators and iterators
   over trees are usually written with an explicit stack.
4. **Is the language hostile to deep recursion?** CPython at 1,000 frames is much stricter than the
   JVM at ~23,000 (§3.3), and neither does tail-call elimination — so a "tail recursive" solution
   is not saved by the runtime in either.

**What is *not* a good reason:** "recursion is slow". The overhead is a constant factor and usually
small; §3 is about *correctness under depth*, not speed. Optimising away a recursion that is
provably $\Theta(\log n)$ deep is effort spent on a non-problem.

**The middle path worth knowing:** keep the recursive version as the readable reference and the
iterative one as the shipped implementation, with a differential test between them — which is
exactly what §1.3 does, 3,000 trees per pair.

</details>

***

### Q12. What is the smallest change that turns DFS into BFS?

<details><summary>Answer</summary>

**Which end of the container you remove from.** §1.3 shows the same function producing both:

```python
pending = deque([root])
while pending:
    n = pending.popleft()   # queue -> breadth-first
    n = pending.pop()       # stack -> depth-first
    ...
```

One method call. Everything else — the loop, the children, the visit — is identical.

**Why this is worth more than a party trick:** it means "depth-first" and "breadth-first" are not
properties of an algorithm, they are properties of a **container policy**. The same skeleton with a
**priority queue** (NB-09) instead becomes best-first search, and with a priority of
"distance so far" it becomes Dijkstra (NB-21). Three famous algorithms, one loop, three containers.

**The detail §1.3 makes explicit:** the stack version does not match `preorder` exactly. Appending
left then right and popping from the back means **right is explored first**, since a stack
reverses. It is a genuine depth-first traversal, mirrored — and `preorder_iter` pushes right first
precisely to cancel that out. If your converted traversal comes out mirrored, this is why.

**And the caveat:** BFS's queue can hold an entire level, which for a balanced tree is $n/2$ nodes
— so **BFS is $\Theta(n)$ space where DFS is $\Theta(h)$**. On a balanced tree DFS is dramatically
cheaper; on a degenerate one it is DFS that costs $\Theta(n)$ and BFS that costs $\Theta(1)$. They
have opposite worst cases, which is the same shape of observation as NB-05 §3.3.

</details>

***

## Coding challenges

### Challenge 1 — a resumable tree iterator

§1.3's iterative traversals run to completion. Make one you can pause.

1. Write an inorder **iterator** class with `next()` and `has_next()`, using an explicit stack, so
   the caller controls the pace. Verify it against `inorder` over thousands of trees.
2. Show it uses $\Theta(h)$ space, not $\Theta(n)$ — it must not flatten the tree up front.
3. Now support **two iterators over the same tree at once**, interleaved, and explain why Morris
   traversal (§1.4) cannot do this at all.
4. Add `peek()` and use two of them to merge two BSTs into one sorted stream in $\Theta(1)$ extra
   space beyond the iterators.

### Challenge 2 — measure the recursion cliff precisely

§3.1 found the failure between 900 and 1,000 nodes. Pin it down.

1. Binary-search the exact degenerate tree size at which `inorder` first raises `RecursionError`,
   and explain why it is somewhat *below* `sys.getrecursionlimit()` — each traversal level costs
   more than one frame.
2. Repeat for `preorder`, `postorder` and `height`, and explain why they differ.
3. Now instrument with `sys.setrecursionlimit` at several values and confirm the cliff moves
   proportionally.
4. Do the same in Java with several thread stack sizes and plot frames against bytes. §3.3's table
   is four points; get enough for a line, and derive the bytes-per-frame.

### Challenge 3 — Morris preorder, and why postorder is worse

§1.4 did Morris **inorder**. The other two are not equally easy.

1. Adapt it to **preorder** — the threading is identical, only the visit moves. Verify output *and*
   tree restoration.
2. Attempt **postorder**. It needs reversing the right spine of a subtree in place and reversing it
   back, which is substantially harder. Implement it, and verify restoration especially carefully.
3. Measure all three against their stack-based equivalents and quantify Morris's constant-factor
   penalty from re-walking spines.
4. Then write the paragraph you would put in a code review explaining when *not* to use Morris.
   §1.4's caveat about abandoned traversals is the important one.

***
# Part 5 - Practice

| # | Exercise | The technique | Difficulty |
|---|---|---|---|
| 1 | Same tree, and symmetric tree | Parallel recursion | ★☆☆☆☆ |
| 2 | Invert a binary tree | The one from the famous tweet | ★☆☆☆☆ |
| 3 | Path sum, all root-to-leaf paths | Carrying state down, backtracking | ★★☆☆☆ |
| 4 | Lowest common ancestor | Postorder returning "what did I find?" | ★★★☆☆ |
| 5 | Build a tree from two traversals | Q8's reconstruction | ★★★★☆ |
| 6 | Maximum path sum | §2.2's pattern, with a sign trap | ★★★★☆ |
| 7 | Flatten a tree to a linked list | In-place pointer surgery | ★★★☆☆ |
| 8 | Count complete-tree nodes in $O(\log^2 n)$ | Exploiting an extra invariant | ★★★★★ |

***

### 1. Same tree, and symmetric tree

- **Brief:** `same(a, b)` recurses on both trees in lockstep. `symmetric(root)` is the same
  function comparing `left` against `right` **mirrored** — `sym(a.left, b.right)` and
  `sym(a.right, b.left)`.
- **Good result:** both $\Theta(n)$, verified against a serialise-and-compare reference (§2.4).
- **The trap:** the null cases. Both empty is `True`; one empty is `False`. Beginners write
  `if a is None or b is None: return False`, which fails on two empty trees. Also do the iterative
  version with a queue of *pairs* — it is a neat use of §2.3.

### 2. Invert a binary tree

- **Brief:** swap every node's children, recursively. Three lines.
- **Good result:** $\Theta(n)$; verify that inverting twice restores the original (a round-trip
  property, like §2.4's).
- **The trap:** none, really — which is the point. It is famous only because a well-known developer
  was rejected for not doing it on a whiteboard. Do the iterative version too, and note that the
  inversion of an inorder traversal is the *reverse* inorder, which is a nice check.

### 3. Path sum, and all root-to-leaf paths

- **Brief:** carry the running sum (and the path) **down** as a parameter, rather than returning it
  up. That is the preorder direction: a node's context is inherited from its parent.
- **Good result:** $\Theta(n)$ for "does a path exist", $\Theta(n \cdot h)$ for collecting all
  paths (output size, unavoidable).
- **The trap:** **backtracking.** If you append to a shared list, you must pop after recursing, or
  siblings inherit each other's paths. Passing `path + [n.val]` avoids it at the cost of a copy per
  node. Know both and know which you wrote. Also: "leaf" means *both* children null — a node with
  one child is not a leaf, and that bug passes many tests.

### 4. Lowest common ancestor

- **Brief:** postorder. Return the node itself if it is `p` or `q`; otherwise recurse both sides.
  If **both** sides return non-null, this node is the LCA; otherwise pass up whichever was non-null.
- **Good result:** $\Theta(n)$, one pass, no parent pointers; verified against a
  collect-both-paths-and-compare reference.
- **The trap:** the version that assumes `p` and `q` are both present. If one may be absent the
  algorithm silently returns the other, which is wrong — you need a second pass or a found-count.
  State which contract you implemented. The BST version (NB-07) is $O(h)$ and much simpler.

### 5. Build a tree from two traversals

- **Brief:** preorder gives the root; find it in inorder to split left from right; recurse. Use a
  **value → index map** for the inorder positions or the lookup makes it $\Theta(n^2)$.
- **Good result:** $\Theta(n)$, verified by round-tripping — build from the traversals of a random
  tree and confirm the fingerprint matches (§2.4's method).
- **The trap:** duplicate values break it entirely, which is why Q8 says "if values are distinct".
  Then try **preorder + postorder** and find the two-node tree it cannot resolve — that failure is
  more instructive than the success.

### 6. Maximum path sum

Largest sum along any path; the path may start and end anywhere and need not touch the root.

- **Brief:** §2.2's pattern exactly — return the best *downward* sum to the parent, accumulate the
  best *through* sum in a side channel.
- **Good result:** $\Theta(n)$, verified against a brute force over all node pairs on small trees.
- **The trap:** **negative values.** A subtree contributing a negative sum should contribute
  **zero** instead — you decline to use it — so the return is `max(0, ...)`. Forget that and one
  negative node poisons every path through it. Also: the answer itself may be negative if all nodes
  are, so the accumulator must not start at 0.

### 7. Flatten a tree to a linked list in place

Rewire into a right-leaning chain in preorder order, using the `right` pointers.

- **Brief:** for each node with a left child, find the **rightmost node of that left subtree**,
  attach the current right subtree to it, then move the left subtree across.
- **Good result:** $\Theta(n)$, $\Theta(1)$ extra space; verify the flattened order equals
  `preorder` of the original, so capture that first.
- **The trap:** this is **Morris-shaped** (§1.4) — the same "find the predecessor and rewire" move
  — except here you do not restore anything, so it is destructive by design. Doing it right after
  §1.4 makes both click.

### 8. Count nodes in a complete tree in $O(\log^2 n)$

- **Brief:** a **complete** tree is filled left to right except possibly the last level. Compare
  the left spine's height with the right spine's: if equal, the left subtree is *perfect* and holds
  $2^h - 1$ nodes with no counting; otherwise the right subtree is perfect. Recurse into the other.
- **Good result:** $O(\log^2 n)$ — $\log n$ levels, each doing an $O(\log n)$ spine walk — verified
  against a plain $\Theta(n)$ count, with the node visits measured to confirm the bound.
- **The trap:** the whole exercise is recognising that **an extra invariant buys a better
  algorithm**. Q10's complaint is that a plain binary tree promises nothing; completeness is a
  promise, and this is what one is worth. It is the same lesson NB-07 applies with the BST
  invariant.

***
# Part 6 - Reading

## Start here

**1. *Introduction to Algorithms* (CLRS), chapters 10.4 and 12.1.**
> 10.4 covers representing rooted trees, including the left-child/right-sibling trick for
> arbitrary branching factors. 12.1 does the traversals with the proofs that inorder on a BST
> yields sorted output — the theorem NB-07 depends on. Short and precise.

**2. [Morris & Knuth on threaded trees]** — J. H. Morris,
*Traversing binary trees simply and cheaply*, Information Processing Letters, 1979.
> Two pages, and it is the origin of §1.4. Worth reading for how carefully it treats the
> restoration step — the property this notebook checks across 3,000 trees — because Morris clearly
> understood that a traversal which mutates its input is only acceptable if the mutation is
> provably undone.

**3. [CPython: `sys.setrecursionlimit` documentation](https://docs.python.org/3/library/sys.html#sys.setrecursionlimit)** —
**Free.**
> Three paragraphs, and it says the thing §3.2 is built on: the limit exists to prevent a **C stack
> overflow and crash**, and setting it too high makes a crash possible. Read it before you ever
> raise the limit in anger.

## The source behind each section

| Section | Where it comes from | Free? |
|---|---|---|
| 1.1 — the recursive definition | **CLRS ch. 10.4**; any text on algebraic data types | 🔍 |
| 1.2 — the four traversals | **Knuth, TAOCP vol. 1 §2.3.1** — the canonical treatment, where the names come from | 🔍 |
| 1.3 — explicit-stack conversion | **CLRS ch. 12.1** exercises; the general recursion-to-iteration transformation | 🔍 |
| 1.4 — **Morris traversal** | **Morris**, *Traversing binary trees simply and cheaply*, IPL 9(5), 1979; **Knuth §2.3.1** on threaded trees | 🔍 |
| 2.2 — return one, track another | Folklore; no canonical source, which is why it is worth naming | — |
| 2.4 — serialisation ambiguity | **Knuth §2.3.1**, exercises on reconstructing trees from traversals | 🔍 |
| 3.1–3.2 — the recursion limit | **CPython** `sys` docs; `Python/ceval.c` on the recursion guard | ✅ |
| 3.3 — JVM stack depth | **JVM Specification** §2.5.2 on the per-thread stack; the `Thread` constructor with `stackSize` | ✅ |
| Q10 — locality | **NB-04 §3.1**; **Drepper**, *What Every Programmer Should Know About Memory* | ✅ |
| Practice 8 — complete trees | The array-embedded tree of **CLRS ch. 6.1** (heaps), which NB-09 builds on | 🔍 |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**Knuth, TAOCP vol. 1 §2.3.1.** It is where preorder, inorder and postorder are named and where
threaded trees — Morris's starting point — are developed. What makes it worth the effort rather
than a modern textbook is that Knuth treats the traversal orders as a *family generated by a
choice*, so the fact that three of them differ by one line's position is derived rather than
observed. §1.2 states that as a fact; Knuth shows why it must be so.

If that is too much, read **Morris's 1979 paper** instead — two pages, and it is the whole of §1.4
including the restoration argument this notebook turned into a test.

Then read the **`sys.setrecursionlimit` docs**, which take two minutes and will stop you doing the
tempting wrong thing in §3.2.

***
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| `RecursionError` on a small tree | Depth is the **height**, not the node count (§3.1) | Explicit stack (§1.3), or balance the tree (NB-08) |
| Works on a million nodes, dies on a thousand | The small tree was degenerate (§3.1) | Ask what bounds the height, not the size |
| Segfault after raising the recursion limit | The limit guards the C stack (§3.2) | Do not raise it far; convert to iteration |
| `StackOverflowError` in Java only in production | Thread stack size differs by deployment (§3.3) | `-Xss`, or an explicit `Thread` stack size — and do not rely on it |
| Iterative traversal comes out mirrored | A stack reverses; push order is backwards (§1.3) | Push right before left for left-first output |
| Iterative postorder visits parents too early | Cannot tell "descending" from "returning" (§1.3) | `last_visited` pointer, or mirrored preorder reversed |
| Tree corrupted after a traversal | Morris abandoned part-way leaves threads in place (§1.4) | Never `break` out of a Morris walk; use an explicit stack |
| Height off by one everywhere | Edge convention mixed with node convention (§2.1) | Pick one; empty = −1 removes the leaf case |
| Diameter misses the longest path | Recursion combined subtree diameters only (§2.2) | Return height, accumulate the answer separately |
| BFS levels bleed together | Queue length not snapshotted (§2.3) | `for _ in range(len(q))` at the top of the loop |
| Deserialised tree has the wrong shape | Format cannot express empty subtrees (§2.4) | Preorder **with explicit nulls**, or preorder + inorder |
| All-paths collection returns duplicates | Shared list not popped after recursing (Practice 3) | Backtrack, or pass `path + [v]` |
| Max path sum poisoned by one negative node | Negative subtree contributions not clamped (Practice 6) | `max(0, child)` before combining |

## Checklist for tree code

- [ ] What bounds the **height**? If it can be $\Theta(n)$, is the traversal iterative (§3.1)?
- [ ] Is the tree shape influenced by **untrusted input** (§3.1)?
- [ ] Is `sys.setrecursionlimit` being used as a fix rather than a stopgap (§3.2)?
- [ ] Is the height convention (edges, empty = −1) applied **consistently** (§2.1)?
- [ ] Does any recursion try to make one return value do two jobs (§2.2)?
- [ ] Does BFS snapshot `len(q)` before draining a level (§2.3)?
- [ ] Can the serialisation format represent an **empty subtree** (§2.4)?
- [ ] Is the round trip tested by comparing **trees**, not strings (§2.4)?
- [ ] If Morris is used: can the traversal ever be abandoned part-way (§1.4)?
- [ ] Is the iterative version differential-tested against the recursive one (§1.3)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `bst_zero_to_hero.ipynb` | The ordering invariant that makes inorder meaningful and search $O(h)$ — Q10's missing promise |
| `balanced_trees_zero_to_hero.ipynb` | Making $h = \Theta(\log n)$, which retires §3's problem entirely |
| `heaps_zero_to_hero.ipynb` | A complete tree stored **in an array** — no pointers, no locality problem |
| [`stacks_queues_zero_to_hero.ipynb`](stacks_queues_zero_to_hero.ipynb) | The stack and queue §1.3 swaps to turn DFS into BFS |

See [`README.md`](README.md) for the full roster and reading order.